# Proyecto SQL: Análisis de datos de una plataforma de lectura

## 1. Descripción del objetivo

## Objetivo
Analizar la base de datos de un servicio de recomendación de libros para extraer insights sobre:
- El volumen de publicaciones recientes.
- La relación entre reseñas y calificaciones.
- El desempeño de editoriales y autores.
- El comportamiento de los usuarios más activos.

Estos hallazgos servirán para fundamentar la propuesta de valor de una nueva aplicación para amantes de la lectura.

## 2. Conexión a la base de datos

In [ ]:
#código para crear una conexión a la base de datos:

# importar librerías
import pandas as pd
from sqlalchemy import create_engine


db_config = {'user': 'practicum_student',         # nombre de usuario
             'pwd': 's65BlTKV3faNIGhmvJVzOqhs', # contraseña
             'host': 'rc1b-wcoijxj3yxfsf3fs.mdb.yandexcloud.net',
             'port': 6432,              # puerto de conexión
             'db': 'data-analyst-final-project-db'}          # nombre de la base de datos

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(db_config['user'],
                                                                     db_config['pwd'],
                                                                       db_config['host'],
                                                                       db_config['port'],
                                                                       db_config['db'])

engine = create_engine(connection_string, connect_args={'sslmode':'require'})

## 3. Exploración inicial de las tablas

In [ ]:
# Exploración inicial de las tablas
tablas = ['books', 'authors', 'publishers', 'ratings', 'reviews']
for tabla in tablas:
    print(f"\n Primeras filas de la tabla: {tabla}")
    df_temp = pd.io.sql.read_sql(f"SELECT * FROM {tabla} LIMIT 5", con=engine)
    count_query = f"SELECT COUNT(*) FROM {tabla}"
    count = pd.io.sql.read_sql(count_query, con=engine).iloc[0,0]
    print(f"Total registros en {tabla}: {count}")
    display(df_temp)


 Primeras filas de la tabla: books
Total registros en books: 1000


,book_id,author_id,title,num_pages,publication_date,publisher_id
0,1,546,'Salem's Lot,594,2005-11-01,93
1,2,465,1 000 Places to See Before You Die,992,2003-05-22,336
2,3,407,13 Little Blue Envelopes (Little Blue Envelope...,322,2010-12-21,135
3,4,82,1491: New Revelations of the Americas Before C...,541,2006-10-10,309
4,5,125,1776,386,2006-07-04,268



 Primeras filas de la tabla: authors
Total registros en authors: 636


,author_id,author
0,1,A.S. Byatt
1,2,Aesop/Laura Harris/Laura Gibbs
2,3,Agatha Christie
3,4,Alan Brennert
4,5,Alan Moore/David Lloyd



 Primeras filas de la tabla: publishers
Total registros en publishers: 340


,publisher_id,publisher
0,1,Ace
1,2,Ace Book
2,3,Ace Books
3,4,Ace Hardcover
4,5,Addison Wesley Publishing Company



 Primeras filas de la tabla: ratings
Total registros en ratings: 6456


,rating_id,book_id,username,rating
0,1,1,ryanfranco,4
1,2,1,grantpatricia,2
2,3,1,brandtandrea,5
3,4,2,lorichen,3
4,5,2,mariokeller,2



 Primeras filas de la tabla: reviews
Total registros en reviews: 2793


,review_id,book_id,username,text
0,1,1,brandtandrea,Mention society tell send professor analysis. ...
1,2,1,ryanfranco,Foot glass pretty audience hit themselves. Amo...
2,3,2,lorichen,Listen treat keep worry. Miss husband tax but ...
3,4,3,johnsonamanda,Finally month interesting blue could nature cu...
4,5,3,scotttamara,Nation purpose heavy give wait song will. List...


- Se visualiza que las tablas tienen la estrcutura que se visualiza en el esquema de base de datos con las claves necesarias para unirlas.

## 4. Consulta SQL para cada una de las tareas.

### Tarea 1: Libros publicados después del 1 de enero de 2000

In [38]:
# Encuentra el número de libros publicados después del 1 de enero de 2000.

# 1. Definimos la consulta
mi_query = "SELECT count(*) as cantidad FROM books WHERE publication_date > '2000-01-01'  "

# 3. Traemos los datos al DataFrame
df = pd.io.sql.read_sql(mi_query, con=engine)

# Imprimir la cantidad de libros publicados después del 1 de enero de 2000.
print(f"La cantidad de libros publicados después del 1 de enero de 2000 es: {df['cantidad'][0]} libros")

La cantidad de libros publicados después del 1 de enero de 2000 es: 819 libros


#### Conclusion:

Se identificaron **819 libros publicados a partir del año 2000**, lo que constituye una base de datos actualizada y relevante. Para la nueva aplicación, esto significa que podemos ofrecer a los usuarios **recomendaciones de títulos recientes**, aumentando la probabilidad de engagement. Además, el volumen es suficiente para entrenar modelos de filtrado colaborativo sin quedarnos cortos en datos.

### Tarea 2: Número de reseñas y calificación promedio por libro

In [42]:
mi_query = """
SELECT 
    b.title AS titulo,
    COALESCE(rev.total_resenas, 0) AS total_resenas,
    rat.promedio_calificacion
FROM books b
LEFT JOIN (
    SELECT book_id, COUNT(review_id) AS total_resenas 
    FROM reviews 
    GROUP BY book_id
) rev ON b.book_id = rev.book_id
LEFT JOIN (
    SELECT book_id, AVG(rating) AS promedio_calificacion 
    FROM ratings 
    GROUP BY book_id
) rat ON b.book_id = rat.book_id
ORDER BY total_resenas DESC;
"""

df = pd.io.sql.read_sql(mi_query, con=engine)

print("--- Resumen de libros ---")
for index, fila in df.head().iterrows():
    print(f"El libro '{fila['titulo']}' tiene {fila['total_resenas']} reseñas y calificación promedio de {fila['promedio_calificacion']:.2f}")

print("\n--- Insights Destacados ---")

# Mayor calificación (solo libros con al menos una calificación)
df_con_rating = df.dropna(subset=['promedio_calificacion'])
top_rating = df_con_rating.sort_values('promedio_calificacion', ascending=False).iloc[0]
print(f"Libro con mayor calificación: '{top_rating['titulo']}' : {top_rating['promedio_calificacion']:.2f}")

# Más reseñas
top_reviews = df.sort_values('total_resenas', ascending=False).iloc[0]
print(f"Libro con más reseñas: '{top_reviews['titulo']}' : {top_reviews['total_resenas']} reseñas")

--- Resumen de libros ---
El libro 'Twilight (Twilight  #1)' tiene 7 reseñas y calificación promedio de 3.66
El libro 'The Curious Incident of the Dog in the Night-Time' tiene 6 reseñas y calificación promedio de 4.08
El libro 'Harry Potter and the Chamber of Secrets (Harry Potter  #2)' tiene 6 reseñas y calificación promedio de 4.29
El libro 'The Da Vinci Code (Robert Langdon  #2)' tiene 6 reseñas y calificación promedio de 3.83
El libro 'The Glass Castle' tiene 6 reseñas y calificación promedio de 4.21

--- Insights Destacados ---
Libro con mayor calificación: 'Misty of Chincoteague (Misty  #1)' : 5.00
Libro con más reseñas: 'Twilight (Twilight  #1)' : 7 reseñas


#### Conclusión

El libro con **mayor calificación promedio** (5.00) es *Misty of Chincoteague*, pero tiene pocas reseñas. Por otro lado, *Twilight* acumula **7 reseñas** (la más alta), pero su calificación es moderada (3.66). Esto revela que **la popularidad no es sinónimo de satisfacción**. Para la app, es clave equilibrar ambos indicadores: mostrar tanto los más votados como los más comentados, y quizás crear una métrica híbrida (ej. "top trending").

### Tarea 3: Editorial con más libros (>50 páginas)

In [23]:
"""
Identifica la editorial que ha publicado el mayor número de libros con más de 50 páginas 
(esto te ayudará a excluir folletos y publicaciones similares de tu análisis).

"""
# 1. Definimos la consulta
mi_query = """ 
SELECT 
    publishers.publisher AS editorial,
    COUNT(books.book_id) AS total_libros
FROM books
INNER JOIN publishers ON books.publisher_id = publishers.publisher_id
WHERE books.num_pages > 50
GROUP BY publishers.publisher
ORDER BY total_libros DESC
LIMIT 1;
"""

# 3. Traemos los datos al DataFrame
df = pd.io.sql.read_sql(mi_query, con=engine)

# 4 Imprimimos el resultado
if not df.empty:
    editorial = df.iloc[0]['editorial']
    cantidad = df.iloc[0]['total_libros']
    print(f"La editorial con más libros (>50 págs) es '{editorial}' con {cantidad} títulos.")

La editorial con más libros (>50 págs) es 'Penguin Books' con 42 títulos.


#### Conclusión

**Penguin Books** es la editorial con más títulos extensos (>50 páginas): 42 libros. Esto la convierte en un **socio estratégico** para la aplicación. Se podría negociar acceso anticipado a sus novedades, o crear una sección destacada "Penguin Recomienda". También sugiere que los usuarios de la plataforma valoran contenido de profundidad (no solo folletos).

### Tarea 4: Autor con mayor calificación promedio (mínimo 50 calificaciones)

In [63]:
# 1. Definimos la consulta del autor destacado
query_autor = """
SELECT a.author as autor, AVG(r.rating) as rating_promedio
FROM books b
JOIN authors a ON b.author_id = a.author_id
JOIN ratings r ON b.book_id = r.book_id
WHERE b.book_id IN (
    SELECT book_id FROM ratings GROUP BY book_id HAVING COUNT(*) >= 50
)
GROUP BY a.author
ORDER BY rating_promedio DESC
LIMIT 1;
"""

# 2. Ejecutamos
df_autor = pd.io.sql.read_sql(query_autor, con=engine)

# 3. Presentación de resultados
if not df_autor.empty:
    autor_top = df_autor.iloc[0]['autor']
    score = df_autor.iloc[0]['rating_promedio']
    print(f"Autor destacado: {autor_top}")
    print(f"Calificación promedio: {score:.2f}")   


Autor destacado: J.K. Rowling/Mary GrandPré
Calificación promedio: 4.29


#### Conclusión

El autor **J.K. Rowling/Mary GrandPré**  alcanza un rating promedio de **4.29** con al menos 50 calificaciones. Esto demuestra una **alta fidelidad de los lectores** hacia su obra. Para la app, podemos aprovechar este insight para:
- Crear una colección "Lecturas imprescindibles" con autores de alta calificación.
- Usar estos autores como gancho en campañas de suscripción.
- Analizar qué elementos comunes tienen estos autores (género, extensión, editorial) para recomendar otros similares.

## Tarea 5: Promedio de reseñas de texto entre usuarios que calificaron más de 50 libros

In [64]:
# Tarea 5 original (promedio para usuarios con >50 calificaciones)
query_activos = """
SELECT 
    AVG(total_reviews) AS promedio_resenas_activos
FROM (
    SELECT 
        r.username,
        COUNT(rv.review_id) AS total_reviews
    FROM ratings r
    LEFT JOIN reviews rv ON r.book_id = rv.book_id AND r.username = rv.username
    WHERE r.username IN (
        SELECT username
        FROM ratings
        GROUP BY username
        HAVING COUNT(book_id) > 50
    )
    GROUP BY r.username
) AS usuarios_activos;
"""

df_activos = pd.io.sql.read_sql(query_activos, con=engine)
promedio_activos = df_activos.iloc[0, 0]

# Promedio para TODOS los usuarios
query_todos = """
SELECT 
    AVG(total_reviews) AS promedio_resenas_todos
FROM (
    SELECT 
        r.username,
        COUNT(rv.review_id) AS total_reviews
    FROM ratings r
    LEFT JOIN reviews rv ON r.book_id = rv.book_id AND r.username = rv.username
    GROUP BY r.username
) AS todos_usuarios;
"""

df_todos = pd.io.sql.read_sql(query_todos, con=engine)
promedio_todos = df_todos.iloc[0, 0]

# Resultados
print(f" Usuarios con >50 calificaciones → promedio de reseñas: {promedio_activos:.2f}")
print(f" Todos los usuarios → promedio de reseñas: {promedio_todos:.2f}")
print(f" Los súper activos escriben {promedio_activos / promedio_todos:.1f} veces más reseñas que la media.")

 Usuarios con >50 calificaciones → promedio de reseñas: 24.33
 Todos los usuarios → promedio de reseñas: 17.46
 Los súper activos escriben 1.4 veces más reseñas que la media.


#### Conclusión

Los usuarios que han calificado **más de 50 libros** escriben en promedio **24.33 reseñas de texto** cada uno. Esto indica que **los lectores más activos también son los que más contribuyen con contenido cualitativo**. Para la aplicación, este grupo es valiosísimo: se les puede incentivar con insignias de "crítico experto" o darles acceso beta a nuevas funciones.

## 5. Conclusión General del Proyecto

El análisis de la base de datos de libros, calificaciones, reseñas, autores y editoriales ha permitido extraer **cuatro insights clave** para la nueva aplicación de lectura:

1. **Volumen relevante de contenido reciente** (819 libros post-2000) → Podemos enfocar la app en tendencias actuales.
2. **Popularidad ≠ satisfacción** → La app debe mostrar tanto los libros más comentados como los mejor calificados y ofrecer filtros combinados.
3. **Penguin Books como aliado estratégico** → 42 títulos extensos; ideal para secciones patrocinadas o colecciones curadas.
4. **Usuarios súper activos (más de 50 calificaciones) son también críticos textuales** (promedio 24 reseñas) → Fuente de contenido generado por el usuario.

### Recomendaciones para el producto

- **Dashboard de descubrimiento**: Tablero con dos pestañas: "Más comentados" (mayor número de reseñas) y "Mejor valorados" (mayor rating con mínimo de reseñas).
- **Alianza editorial**: Negociar con Penguin Books una sección destacada dentro de la app a cambio de visibilidad.
- **Gamificación para usuarios activos**: Crear el nivel "Crítico Élite" para quienes superen 50 calificaciones y ofrecerles recompensas (descuentos, menciones).

Con estos hallazgos, la propuesta de valor de la aplicación se fundamenta en **datos reales** y se orienta a resolver necesidades concretas de los lectores más comprometidos.